In [12]:
from google.colab import files

uploads = files.upload()

Saving advanced_student_performance_v3.csv to advanced_student_performance_v3.csv


In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error
from sklearn.impute import SimpleImputer

In [13]:
df = pd.read_csv('advanced_student_performance_v3.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 22 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Math_Grade              1000 non-null   int64  
 1   Science_Grade           1000 non-null   int64  
 2   History_Grade           1000 non-null   int64  
 3   English_Grade           1000 non-null   int64  
 4   Art_Grade               1000 non-null   int64  
 5   Computer_Science_Grade  1000 non-null   int64  
 6   Study_Hours_Per_Week    1000 non-null   float64
 7   School_Track            1000 non-null   object 
 8   Attendance_Tier         1000 non-null   object 
 9   Score_Jan               1000 non-null   float64
 10  Score_Feb               1000 non-null   float64
 11  Score_Mar               1000 non-null   float64
 12  Score_Apr               1000 non-null   float64
 13  Score_May               1000 non-null   float64
 14  Score_Jun               1000 non-null   f

In [14]:
df.head()

,Math_Grade,Science_Grade,History_Grade,English_Grade,Art_Grade,Computer_Science_Grade,Study_Hours_Per_Week,School_Track,Attendance_Tier,Score_Jan,...,Score_Apr,Score_May,Score_Jun,Score_Jul,Score_Aug,Score_Sep,Score_Oct,Score_Nov,Score_Dec,Overall_Final_Grade
0,88,82,52,56,68,94,14.8,Business,Low,72.4,...,75.3,73.8,75.4,78.1,79.3,77.2,77.2,78.1,79.2,79.2
1,78,71,55,53,74,66,7.3,Arts,High,66.8,...,68.9,68.6,68.6,69.5,71.6,70.5,73.2,73.6,73.2,73.1
2,64,70,58,78,71,89,13.6,Business,High,73.2,...,74.5,74.9,75.8,76.3,79.2,79.2,78.7,79.9,80.3,80.1
3,92,55,55,63,80,70,18.8,Tech,Medium,67.8,...,71.9,70.6,71.7,69.3,71.4,72.0,71.4,73.1,72.8,73.4
4,57,55,58,52,66,51,24.0,Science,High,58.2,...,60.7,61.5,61.9,64.3,64.1,65.1,64.0,65.3,65.7,66.9


In [15]:
x = df.drop('Overall_Final_Grade', axis=1)
y = df['Overall_Final_Grade']

In [9]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [16]:
numerical_features = x_train.select_dtypes(include=['int64', 'float64']).columns
categorical_features = x_train.select_dtypes(include=['object']).columns

In [17]:
print(numerical_features)
print(categorical_features)

Index(['Math_Grade', 'Science_Grade', 'History_Grade', 'English_Grade',
       'Art_Grade', 'Computer_Science_Grade', 'Study_Hours_Per_Week',
       'Score_Jan', 'Score_Feb', 'Score_Mar', 'Score_Apr', 'Score_May',
       'Score_Jun', 'Score_Jul', 'Score_Aug', 'Score_Sep', 'Score_Oct',
       'Score_Nov', 'Score_Dec'],
      dtype='object')
Index([], dtype='object')


In [10]:
sc = StandardScaler()
oe = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
num_transformer = Pipeline(
    steps=[
      ('imputer', SimpleImputer(strategy='mean')),
      ('scaler', sc)
     ]
    )

cat_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', oe)
    ]
    )

In [18]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, numerical_features),
        ('cat', cat_transformer, categorical_features)
    ]
)

In [19]:
x_train_preprocessed = preprocessor.fit_transform(x_train)
y_train_preprocessed = np.log1p(y_train)
x_test_preprocessed = preprocessor.transform(x_test)
y_test_preprocessed = np.log1p(y_test.values)

In [20]:
x_train_tensor = torch.FloatTensor(x_train_preprocessed)
x_test_tensor = torch.FloatTensor(x_test_preprocessed)
y_train_tensor = torch.FloatTensor(y_train_preprocessed.values).reshape(-1, 1)
y_test_tensor = torch.FloatTensor(y_test_preprocessed).reshape(-1, 1)

In [21]:

print("\n✅ Tensors created from PREPROCESSED data!")
print(f"x_train_tensor: {x_train_tensor.shape}")
print(f"x_test_tensor: {x_test_tensor.shape}")
print(f"y_train_tensor: {y_train_tensor.shape}")
print(f"y_test_tensor: {y_test_tensor.shape}")


✅ Tensors created from PREPROCESSED data!
x_train_tensor: torch.Size([800, 19])
x_test_tensor: torch.Size([200, 19])
y_train_tensor: torch.Size([800, 1])
y_test_tensor: torch.Size([200, 1])


In [22]:
class HomePrices(nn.Module):
  def __init__(self, input_features):
    super().__init__()
    self.layer1 = nn.Linear(input_size, 32)
    self.layer2 = nn.Linear(32, 16)
    self.layer3 = nn.Linear(16, 8)
    self.layer4 = nn.Linear(8, 4)
    self.layer5 = nn.Linear(4, 2)
    self.layer6 = nn.Linear(2, 1)
    self.dropout = nn.Dropout(0.2)
  def forward(self, x):
    x = torch.relu(self.layer1(x))
    x = self.dropout(x)
    x = torch.relu(self.layer2(x))
    x = self.dropout(x)
    x = torch.relu(self.layer3(x))
    x = self.dropout(x)
    x = torch.relu(self.layer4(x))
    x = self.dropout(x)
    x = torch.relu(self.layer5(x))
    x = self.dropout(x)
    x = self.layer6(x)

    return x

In [23]:
input_size = x_train_tensor.shape[1]
model = HomePrices(input_size)

In [24]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

In [25]:
num_epoch = 200
batch_size = 64
train_loss = []
test_losses = []

In [26]:
for i in range(num_epoch):
  model.train()
  epoch_loss = 0
  for j in range(0, len(x_train_tensor), batch_size):
    batch_x = x_train_tensor[j:j+batch_size]
    batch_y = y_train_tensor[j:j+batch_size]
    outputs = model(batch_x)
    loss = criterion(outputs, batch_y)
    loss.backward()
    optimizer.zero_grad()
    optimizer.step()
    epoch_loss += loss.item()
  avg_train_loss = epoch_loss / (len(x_train_tensor) / batch_size)


  with torch.no_grad():
    test_outputs = model(x_test_tensor)
    test_loss = criterion(test_outputs, y_test_tensor)
    test_losses.append(test_loss.item())

    if (i+1)%20==0:
      print(f'Epoch{i+1} / {num_epoch} Train loss:{avg_train_loss} Test loss{test_loss.item()}')

  model.eval()
  with torch.no_grad():
    train_outputs = model(x_train_tensor)
    train_loss = criterion(train_outputs, y_train_tensor)
    train_mse = mean_squared_error(train_outputs.numpy(), y_train_tensor.numpy())
    train_rmse = np.sqrt(train_mse)
    test_outputs = model(x_test_tensor)
    test_loss = criterion(test_outputs, y_test_tensor)
    test_mse = mean_squared_error(test_outputs.numpy(), y_test_tensor.numpy())
    test_rmse = np.sqrt(test_mse)

    print("Training loss: ", train_loss.item())
    print("Training MSE: ", train_mse)
    print("Training RMSE: ", train_rmse)
    print("Test loss: ", test_loss.item())
    print("Test MSE: ", test_mse)
    print("Test RMSE: ", test_rmse)

Training loss:  13.511091232299805
Training MSE:  13.511092185974121
Training RMSE:  3.675743759564059
Test loss:  13.513903617858887
Test MSE:  13.513903617858887
Test RMSE:  3.6761261700136036
Training loss:  13.511091232299805
Training MSE:  13.511092185974121
Training RMSE:  3.675743759564059
Test loss:  13.513903617858887
Test MSE:  13.513903617858887
Test RMSE:  3.6761261700136036
Training loss:  13.511091232299805
Training MSE:  13.511092185974121
Training RMSE:  3.675743759564059
Test loss:  13.513903617858887
Test MSE:  13.513903617858887
Test RMSE:  3.6761261700136036
Training loss:  13.511091232299805
Training MSE:  13.511092185974121
Training RMSE:  3.675743759564059
Test loss:  13.513903617858887
Test MSE:  13.513903617858887
Test RMSE:  3.6761261700136036
Training loss:  13.511091232299805
Training MSE:  13.511092185974121
Training RMSE:  3.675743759564059
Test loss:  13.513903617858887
Test MSE:  13.513903617858887
Test RMSE:  3.6761261700136036
Training loss:  13.511091